# Regime-Based Bitcoin Accumulation Strategy

This notebook implements the regime-based strategy.
The strategy uses StackSats MVRV, StackSats Momentum, custom Smooth Moving Average model and Xgboost model as candidate strategies. DCA is used as the benchmark.

In [1]:
# ============================================================
# Cell 1: Imports, data preparation check, and configuration
# ============================================================
# This cell imports required libraries, checks whether the prepared
# StackSats BTC analytics dataset exists, prepares it if missing,
# and defines all strategy settings.

import polars as pl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import subprocess

from stacksats.runner.core import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.mvrv.core import MVRVStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy

# XGBoost import.
# If this fails, install it in your environment:
# pip install xgboost
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "xgboost is not installed. Install it using: pip install xgboost "
        "or add `xgboost` to your environment.yml under pip or conda dependencies."
    ) from exc


# ============================================================
# StackSats prepared dataset check
# ============================================================
# pip install stacksats installs the package, but it does not automatically
# create ~/.stacksats/data/bitcoin_analytics.parquet.
# This block prepares the file if it is missing.

btc_path = Path.home() / ".stacksats" / "data" / "bitcoin_analytics.parquet"

# IMPORTANT:
# Update this path if your brk_metrics.parquet is in a different location.
# If this notebook is inside the notebooks/ folder and data/ is at repo root,
# then ../data/brk_metrics.parquet is usually correct.
raw_brk_path = Path("../data/brk_metrics.parquet")

if not btc_path.exists():
    print(f"Prepared dataset not found at: {btc_path}")
    print("Preparing StackSats analytics dataset...")

    if not raw_brk_path.exists():
        raise FileNotFoundError(
            f"Raw BRK metrics file not found at: {raw_brk_path}. "
            "Please update raw_brk_path to the correct location of brk_metrics.parquet."
        )

    subprocess.run(
        [
            "stacksats",
            "data",
            "prepare",
            "--source",
            str(raw_brk_path),
        ],
        check=True,
    )

if not btc_path.exists():
    raise FileNotFoundError(
        f"Failed to create prepared dataset at {btc_path}."
    )

print(f"Using prepared dataset: {btc_path}")


# ============================================================
# Strategy configuration
# ============================================================

# Budget used per 365-day window.
TOTAL_BUDGET_USD = 1000.0

# Train and test periods.
TRAIN_START = "2018-01-01"
TRAIN_END = "2023-12-31"
TEST_START = "2024-01-01"
TEST_END = "2025-12-31"

# Each evaluation window is 365 days.
WINDOW_SIZE = 365

# StackSats-style exponential decay factor for percentile aggregation.
# 0.9 means each older window receives 90% of the weight of the next newer window.
EXP_DECAY_FACTOR = 0.90


# Lookbacks used for momentum, SMA, drawdown, and regime classification.
MOMENTUM_LOOKBACK = 60
SMA_LOOKBACK = 90
DRAWDOWN_LOOKBACK = 180
REGIME_LOOKBACK = 180

# Unique lookback values used to create rolling features.
LOOKBACK_DAYS = sorted({
    MOMENTUM_LOOKBACK,
    SMA_LOOKBACK,
    DRAWDOWN_LOOKBACK,
    REGIME_LOOKBACK,
})

# Small floor to avoid zero or negative allocation signals.
SIGNAL_FLOOR = 1e-8

# Minimum number of days required for a regime to be evaluated.
MIN_REGIME_DAYS = 20

# Tolerance used to decide whether a result is better, worse, or tied.
STATUS_TOLERANCE_PCT = 1e-6

# XGBoost target settings.
# The model predicts whether future 30-day return is in the top 30%
# of recent rolling opportunity.
XGB_FORWARD_HORIZON = 30
XGB_TARGET_ROLLING_WINDOW = 365
XGB_TARGET_QUANTILE = 0.70

# XGBoost signal-to-weight setting.
# Higher values make the ML candidate more aggressive.
XGB_SIGNAL_STRENGTH = 2.00

# Fallback strategy used if a regime appears in test but was not seen in training.
FALLBACK_STRATEGY = f"sma_{SMA_LOOKBACK}d_weight"

# Candidate strategies used in regime selection.
# DCA is benchmark only and is not selected as a candidate here.
# No valuation_weight, adaptive softmax, or LSTM is used in this notebook.
CANDIDATE_COLS = [
    "stacksats_mvrv_weight",
    "stacksats_momentum_weight",
    f"sma_{SMA_LOOKBACK}d_weight",
    "xgboost_weight",
]

Using prepared dataset: C:\Users\ragha\.stacksats\data\bitcoin_analytics.parquet


In [2]:
# ============================================================
# Cell 2: Helper functions
# ============================================================
# This cell defines general helper functions used across the notebook.
# These functions are reused for labeling results, formatting chart text,
# trimming complete windows, normalizing weights, and summarizing results.

def get_status_from_pct_diff(pct_diff, tolerance=STATUS_TOLERANCE_PCT):
    """
    Convert percentage improvement into a simple status label.

    Parameters
    ----------
    pct_diff : float
        Percentage difference of strategy performance versus DCA.
    tolerance : float
        Small threshold used to avoid classifying tiny numerical differences
        as meaningful wins or losses.

    Returns
    -------
    str
        'better' if strategy beats DCA, 'worse' if it underperforms,
        and 'tie' if the difference is within tolerance.
    """
    if pct_diff > tolerance:
        return "better"
    if pct_diff < -tolerance:
        return "worse"
    return "tie"


def format_arrow_text(extra_spd, improvement_pct):
    """
    Create chart annotation text for positive or negative SPD improvement.

    Parameters
    ----------
    extra_spd : float
        Extra sats per dollar compared with DCA.
    improvement_pct : float
        Percentage improvement compared with DCA.

    Returns
    -------
    tuple[str, str]
        Formatted text and color name.
    """
    if extra_spd >= 0:
        return f"▲ +{extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "green"
    return f"▼ {extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "red"


def trim_full_windows(df: pd.DataFrame, window_size: int = WINDOW_SIZE):
    """
    Keep only complete fixed-length windows.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe sorted by date.
    window_size : int
        Number of rows per evaluation window.

    Returns
    -------
    tuple[pd.DataFrame, int, int]
        Trimmed dataframe, number of complete windows, and number of dropped rows.
    """
    n_full = len(df) // window_size
    n_eval = n_full * window_size
    remainder = len(df) - n_eval
    return df.iloc[:n_eval].copy(), n_full, remainder


def build_simple_normalized_weights(signal_multiplier, signal_floor=SIGNAL_FLOOR):
    """
    Convert signal multipliers into normalized daily allocation weights.

    Parameters
    ----------
    signal_multiplier : array-like
        Raw signal strength or multiplier values.
    signal_floor : float
        Minimum allowed signal value to prevent zero or negative weights.

    Returns
    -------
    np.ndarray
        Daily allocation weights that sum to 1.
    """
    signal_multiplier = np.asarray(signal_multiplier, dtype=float)

    if len(signal_multiplier) == 0:
        raise ValueError("Empty signal array.")

    clean_signal = np.nan_to_num(
        signal_multiplier,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    clean_signal = np.maximum(clean_signal, signal_floor)

    if clean_signal.sum() <= 0:
        return np.full(len(clean_signal), 1.0 / len(clean_signal))

    return clean_signal / clean_signal.sum()


def compute_stack_sats_exp_decay_average(
    percentile_values,
    decay_factor=EXP_DECAY_FACTOR,
):
    """
    Calculate StackSats-style exponentially decayed average percentile.

    Parameters
    ----------
    percentile_values : array-like
        Chronologically ordered percentile values.
        Older values should come first and newer values should come last.
    decay_factor : float
        Exponential decay factor.

    Returns
    -------
    float
        Exponentially decayed average percentile.

        exp_weights = 0.9 ** np.arange(N - 1, -1, -1)
        exp_weights /= exp_weights.sum()
        exp_avg_pct = (percentile_values * exp_weights).sum()

    This gives the newest window the largest weight and older windows
    gradually lower weights.
    """
    values = np.asarray(percentile_values, dtype=float)

    if len(values) == 0:
        return np.nan

    values = np.nan_to_num(
        values,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    n = len(values)

    exp_weights = decay_factor ** np.arange(n - 1, -1, -1)
    exp_weights = exp_weights / exp_weights.sum()

    return float((values * exp_weights).sum())


def add_spd_percentiles_to_window_summary(window_summary_df):
    """
    Add dynamic and uniform SPD percentile columns.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary with strategy_spd and dca_spd.

    Returns
    -------
    pd.DataFrame
        Window summary with:
        - dynamic_percentile
        - uniform_percentile

    Notes
    -----
    Here, percentiles are calculated across the available 365-day windows
    in the current period.
    """
    out = window_summary_df.copy().sort_values("start_date").reset_index(drop=True)

    out["dynamic_percentile"] = (
        out["strategy_spd"]
        .rank(method="average", pct=True)
        * 100.0
    )

    out["uniform_percentile"] = (
        out["dca_spd"]
        .rank(method="average", pct=True)
        * 100.0
    )

    return out


def calculate_stack_sats_exp_decay_metrics(window_summary_df):
    """
    Calculate exp-decay percentile metrics for a window summary.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary with dynamic_percentile and uniform_percentile.

    Returns
    -------
    dict
        Dictionary with exp_decay_percentile and uniform_exp_decay_percentile.
    """
    ordered = window_summary_df.copy().sort_values("start_date").reset_index(drop=True)

    exp_decay_percentile = compute_stack_sats_exp_decay_average(
        ordered["dynamic_percentile"].to_numpy(),
        decay_factor=EXP_DECAY_FACTOR,
    )

    uniform_exp_decay_percentile = compute_stack_sats_exp_decay_average(
        ordered["uniform_percentile"].to_numpy(),
        decay_factor=EXP_DECAY_FACTOR,
    )

    return {
        "exp_decay_percentile": exp_decay_percentile,
        "uniform_exp_decay_percentile": uniform_exp_decay_percentile,
    }




def summarize_spd_like_composite(window_summary_df):
    """
    Summarize performance across all 365-day windows.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary dataframe containing strategy_sats, dca_sats,
        strategy_spd, dca_spd, and result columns.

    Returns
    -------
    dict
        Summary metrics including total sats, SPD sums, improvement percentage,
        wins, losses, ties, and win rate.
    """
    n_windows = len(window_summary_df)

    strategy_spd_sum = window_summary_df["strategy_spd"].sum()
    dca_spd_sum = window_summary_df["dca_spd"].sum()
    extra_spd_sum = strategy_spd_sum - dca_spd_sum

    spd_ratio = strategy_spd_sum / dca_spd_sum
    improvement_pct = (spd_ratio - 1.0) * 100.0

    strategy_sats = window_summary_df["strategy_sats"].sum()
    dca_sats = window_summary_df["dca_sats"].sum()
    extra_sats = strategy_sats - dca_sats

    wins = int((window_summary_df["result"] == "better").sum())
    losses = int((window_summary_df["result"] == "worse").sum())
    ties = int((window_summary_df["result"] == "tie").sum())

    win_rate_pct = wins / n_windows * 100.0 if n_windows > 0 else 0.0

    if {"dynamic_percentile", "uniform_percentile"}.issubset(window_summary_df.columns):
        exp_decay_metrics = calculate_stack_sats_exp_decay_metrics(window_summary_df)
    else:
        exp_decay_metrics = {
            "exp_decay_percentile": np.nan,
            "uniform_exp_decay_percentile": np.nan,
        }

    return {
        "n_windows": n_windows,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "win_rate_pct": win_rate_pct,
        "exp_decay_percentile": exp_decay_metrics.get("exp_decay_percentile", np.nan),
        "uniform_exp_decay_percentile": exp_decay_metrics.get("uniform_exp_decay_percentile", np.nan),

        "strategy_sats": strategy_sats,
        "dca_sats": dca_sats,
        "extra_sats_vs_dca": extra_sats,

        "strategy_spd_sum": strategy_spd_sum,
        "dca_spd_sum": dca_spd_sum,
        "extra_spd_sum_vs_dca": extra_spd_sum,

        "strategy_spd_avg": strategy_spd_sum / n_windows,
        "dca_spd_avg": dca_spd_sum / n_windows,
        "extra_spd_avg_vs_dca": extra_spd_sum / n_windows,

        "spd_ratio": spd_ratio,
        "improvement_pct": improvement_pct,
    }


def build_log_tick_values_and_text():
    """
    Build custom y-axis tick values and labels for the BTC log-price chart.

    Returns
    -------
    tuple[list[int], list[str]]
        Tick values and corresponding display labels.
    """
    tickvals = [
        3000, 4000, 5000, 6000, 7000, 8000, 9000,
        10000,
        20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000,
        100000,
    ]

    ticktext = [
        "3", "4", "5", "6", "7", "8", "9",
        "10k",
        "2", "3", "4", "5", "6", "7", "8", "9",
        "100k",
    ]

    return tickvals, ticktext

In [3]:
# ============================================================
# Cell 3: Load BTC data
# ============================================================
# This cell loads the prepared Bitcoin analytics parquet file.
# It checks that all required columns exist before moving forward.

if not btc_path.exists():
    raise FileNotFoundError(f"Could not find: {btc_path}")

btc_df = (
    pl.read_parquet(btc_path)
    .with_columns(pl.col("date").cast(pl.Datetime))
    .sort("date")
)

required_cols = [
    "date",
    "price_usd",
    "mvrv",
    "adjusted_sopr",
    "adjusted_sopr_7d_ema",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

missing_cols = [col for col in required_cols if col not in btc_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns from bitcoin_analytics.parquet: {missing_cols}")

print("Loaded BTC rows:", btc_df.height)

display(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
    )
)

Loaded BTC rows: 5689


min_date,max_date
datetime[μs],datetime[μs]
2010-08-16 00:00:00,2026-03-13 00:00:00


In [4]:
# ============================================================
# Cell 4: StackSats strategy export setup
# ============================================================
# This cell sets up the StackSats strategy runner.
# It exports daily weights from built-in StackSats MVRV and Momentum strategies.

runner = StrategyRunner()

stacksats_strategy_objects = {
    "stacksats_mvrv_weight": MVRVStrategy(),
    "stacksats_momentum_weight": MomentumStrategy(),
}

# Cache prevents recomputing weights for the same strategy/window repeatedly.
_export_cache = {}


def export_stacksats_weights_for_window(
    strategy_key: str,
    window_df: pd.DataFrame,
    full_btc_df: pl.DataFrame,
):
    """
    Export StackSats strategy weights for one 365-day window.

    Parameters
    ----------
    strategy_key : str
        Key identifying which StackSats strategy to run.
    window_df : pd.DataFrame
        Current 365-day window dataframe.
    full_btc_df : pl.DataFrame
        Full BTC analytics dataframe in Polars format.

    Returns
    -------
    np.ndarray
        Normalized daily weights for the selected StackSats strategy.
    """
    window_start = pd.to_datetime(window_df["date"].min()).strftime("%Y-%m-%d")
    window_end = pd.to_datetime(window_df["date"].max()).strftime("%Y-%m-%d")

    cache_key = (strategy_key, window_start, window_end)

    if cache_key in _export_cache:
        return _export_cache[cache_key].copy()

    config = ExportConfig(
        range_start=window_start,
        range_end=window_end,
    )

    window_btc_df = (
        full_btc_df
        .filter(
            (pl.col("date") >= pd.to_datetime(window_start)) &
            (pl.col("date") <= pd.to_datetime(window_end)) &
            pl.col("price_usd").is_not_null()
        )
        .sort("date")
    )

    if window_btc_df.is_empty():
        raise ValueError(f"No BTC data available for {window_start} to {window_end}")

    strategy = stacksats_strategy_objects[strategy_key]

    export_obj = runner.export(
        strategy,
        config,
        btc_df=window_btc_df,
    )

    weights = export_obj.to_dataframe()

    if not isinstance(weights, pl.DataFrame):
        weights = pl.from_pandas(weights)

    weights = weights.with_columns([
        pl.col("start_date").cast(pl.Datetime),
        pl.col("end_date").cast(pl.Datetime),
        pl.col("date").cast(pl.Datetime),
    ])

    latest_end = weights.select(pl.col("end_date").max()).item()

    one_window = (
        weights
        .filter(pl.col("end_date") == latest_end)
        .sort("date")
        .select(["date", "weight"])
        .rename({"weight": "raw_weight"})
        .to_pandas()
    )

    one_window["date"] = pd.to_datetime(one_window["date"])

    merged = (
        window_df[["date"]]
        .merge(one_window, on="date", how="left")
        .sort_values("date")
        .reset_index(drop=True)
    )

    if merged["raw_weight"].isna().any():
        missing_dates = (
            merged.loc[merged["raw_weight"].isna(), "date"]
            .dt.strftime("%Y-%m-%d")
            .head(10)
            .tolist()
        )

        raise ValueError(
            f"Missing StackSats weights for {strategy_key} "
            f"from {window_start} to {window_end}. "
            f"Example missing dates: {missing_dates}"
        )

    # This function returns the original StackSats normalized weights.
    # Adaptive softmax is applied later in Cell 8.
    final_weights = build_simple_normalized_weights(
        merged["raw_weight"].values
    )

    _export_cache[cache_key] = final_weights.copy()

    return final_weights

In [5]:
# ============================================================
# Cell 5: Feature engineering
# ============================================================
# This cell creates rolling features used for regime classification.
# It creates SMA values, returns, rolling highs, SMA ratios, and drawdown features.

feature_exprs = []

for d in LOOKBACK_DAYS:
    feature_exprs.extend([
        pl.col("price_usd")
        .rolling_mean(window_size=d, min_samples=max(3, int(d * 0.30)))
        .alias(f"price_{d}d_sma"),

        pl.col("price_usd")
        .pct_change(d)
        .alias(f"btc_return_{d}d"),

        pl.col("price_usd")
        .rolling_max(window_size=d, min_samples=max(3, int(d * 0.30)))
        .alias(f"rolling_high_{d}d"),
    ])

btc_df = btc_df.with_columns(feature_exprs)

ratio_exprs = []

for d in LOOKBACK_DAYS:
    ratio_exprs.extend([
        (pl.col("price_usd") / pl.col(f"price_{d}d_sma"))
        .alias(f"price_{d}d_sma_ratio"),

        ((pl.col("price_usd") / pl.col(f"rolling_high_{d}d")) - 1)
        .alias(f"drawdown_{d}d"),
    ])

btc_df = btc_df.with_columns(ratio_exprs)

btc_data = btc_df.to_pandas()
btc_data["date"] = pd.to_datetime(btc_data["date"])

feature_cols = [
    "price_usd",
    "mvrv",
    "adjusted_sopr",
    "adjusted_sopr_7d_ema",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

for d in LOOKBACK_DAYS:
    feature_cols.extend([
        f"price_{d}d_sma",
        f"price_{d}d_sma_ratio",
        f"btc_return_{d}d",
        f"drawdown_{d}d",
    ])

btc_data = (
    btc_data
    .dropna(subset=feature_cols)
    .sort_values("date")
    .reset_index(drop=True)
)

print("Cleaned date range:")
print(btc_data["date"].min(), "to", btc_data["date"].max())
print("Cleaned rows:", len(btc_data))

# ============================================================
# XGBoost feature aliases
# ============================================================
# These alias columns make the XGBoost feature names easier to understand.
# They point to the dynamic columns created from the configured lookbacks.

btc_data["sma_ratio_feature"] = btc_data[f"price_{SMA_LOOKBACK}d_sma_ratio"]

btc_data["momentum_btc_price_return"] = btc_data[f"btc_return_{MOMENTUM_LOOKBACK}d"]

btc_data["sma_period_btc_price_return"] = btc_data[f"btc_return_{SMA_LOOKBACK}d"]

btc_data["drawdown_feature"] = btc_data[f"drawdown_{DRAWDOWN_LOOKBACK}d"]

xgb_feature_description_df = pd.DataFrame([
    {
        "xgb_feature_name": "sma_ratio_feature",
        "source_column": f"price_{SMA_LOOKBACK}d_sma_ratio",
        "source_config": "SMA_LOOKBACK",
        "lookback_days": SMA_LOOKBACK,
        "meaning": "Price divided by SMA. Below 1 means BTC is trading below its moving average.",
    },
    {
        "xgb_feature_name": "momentum_btc_price_return",
        "source_column": f"btc_return_{MOMENTUM_LOOKBACK}d",
        "source_config": "MOMENTUM_LOOKBACK",
        "lookback_days": MOMENTUM_LOOKBACK,
        "meaning": "BTC price return over the momentum lookback period.",
    },
    {
        "xgb_feature_name": "sma_period_btc_price_return",
        "source_column": f"btc_return_{SMA_LOOKBACK}d",
        "source_config": "SMA_LOOKBACK",
        "lookback_days": SMA_LOOKBACK,
        "meaning": "BTC price return over the same period used for the SMA feature.",
    },
    {
        "xgb_feature_name": "drawdown_feature",
        "source_column": f"drawdown_{DRAWDOWN_LOOKBACK}d",
        "source_config": "DRAWDOWN_LOOKBACK",
        "lookback_days": DRAWDOWN_LOOKBACK,
        "meaning": "Current price drawdown from the rolling high over the drawdown lookback period.",
    },
    {
        "xgb_feature_name": "mvrv",
        "source_column": "mvrv",
        "source_config": "on-chain metric",
        "lookback_days": None,
        "meaning": "Valuation metric comparing market value to realized value.",
    },
    {
        "xgb_feature_name": "adjusted_sopr",
        "source_column": "adjusted_sopr",
        "source_config": "on-chain metric",
        "lookback_days": None,
        "meaning": "Spent output profit ratio. Values below 1 can indicate loss-selling.",
    },
    {
        "xgb_feature_name": "adjusted_sopr_7d_ema",
        "source_column": "adjusted_sopr_7d_ema",
        "source_config": "on-chain metric",
        "lookback_days": 7,
        "meaning": "Smoothed SOPR signal.",
    },
    {
        "xgb_feature_name": "realized_cap_growth_rate",
        "source_column": "realized_cap_growth_rate",
        "source_config": "on-chain metric",
        "lookback_days": None,
        "meaning": "Growth rate of realized capitalization.",
    },
    {
        "xgb_feature_name": "market_cap_growth_rate",
        "source_column": "market_cap_growth_rate",
        "source_config": "market metric",
        "lookback_days": None,
        "meaning": "Growth rate of market capitalization.",
    },
])

display(xgb_feature_description_df)

Cleaned date range:
2011-08-16 00:00:00 to 2026-03-13 00:00:00
Cleaned rows: 5324


,xgb_feature_name,source_column,source_config,lookback_days,meaning
0,sma_ratio_feature,price_90d_sma_ratio,SMA_LOOKBACK,90.0,Price divided by SMA. Below 1 means BTC is tra...
1,momentum_btc_price_return,btc_return_60d,MOMENTUM_LOOKBACK,60.0,BTC price return over the momentum lookback pe...
2,sma_period_btc_price_return,btc_return_90d,SMA_LOOKBACK,90.0,BTC price return over the same period used for...
3,drawdown_feature,drawdown_180d,DRAWDOWN_LOOKBACK,180.0,Current price drawdown from the rolling high o...
4,mvrv,mvrv,on-chain metric,NaN,Valuation metric comparing market value to rea...
5,adjusted_sopr,adjusted_sopr,on-chain metric,NaN,Spent output profit ratio. Values below 1 can ...
6,adjusted_sopr_7d_ema,adjusted_sopr_7d_ema,on-chain metric,7.0,Smoothed SOPR signal.
7,realized_cap_growth_rate,realized_cap_growth_rate,on-chain metric,NaN,Growth rate of realized capitalization.
8,market_cap_growth_rate,market_cap_growth_rate,market metric,NaN,Growth rate of market capitalization.


In [6]:
# ============================================================
# Cell 6: Simpler regime classification
# ============================================================
# This cell labels every day with a simpler combined market regime.
# The regime combines:
# 1. BTC trend behavior
# 2. MVRV valuation behavior
# 3. realized-cap vs market-cap growth behavior

def classify_btc_mvrv_market_cap_regime(row):
    """
    Classify each day into a simplified BTC/on-chain regime.

    The regime combines:
    1. BTC trend regime
    2. MVRV valuation regime
    3. market-cap / realized-cap growth regime

    Returns
    -------
    str
        Combined regime label in the format:
        BTC trend regime | MVRV valuation regime | cap-growth regime
    """

    # ------------------------------------------------------------
    # BTC trend features
    # ------------------------------------------------------------
    # price_{SMA_LOOKBACK}d_sma_ratio:
    # Price relative to the selected SMA window.
    # Example: if SMA_LOOKBACK = 90, this is price / 90-day SMA.
    sma_selected_ratio = row[f"price_{SMA_LOOKBACK}d_sma_ratio"]

    # price_{REGIME_LOOKBACK}d_sma_ratio:
    # Price relative to the longer regime SMA window.
    # Example: if REGIME_LOOKBACK = 180, this is price / 180-day SMA.
    sma_regime_ratio = row[f"price_{REGIME_LOOKBACK}d_sma_ratio"]

    # Shorter-term price return used for recovery/momentum behavior.
    return_momentum = row[f"btc_return_{MOMENTUM_LOOKBACK}d"]

    # Medium-term price return aligned with the SMA lookback.
    return_sma = row[f"btc_return_{SMA_LOOKBACK}d"]

    # Drawdown from recent rolling high.
    drawdown_regime = row[f"drawdown_{DRAWDOWN_LOOKBACK}d"]

    # ------------------------------------------------------------
    # On-chain / market features
    # ------------------------------------------------------------
    mvrv = row["mvrv"]
    realized_growth = row["realized_cap_growth_rate"]
    market_growth = row["market_cap_growth_rate"]

    # ------------------------------------------------------------
    # 1. BTC trend regime
    # ------------------------------------------------------------
    if drawdown_regime <= -0.50:
        btc_regime = "BTC Severe Drawdown"

    elif drawdown_regime <= -0.30:
        btc_regime = "BTC Deep Drawdown"

    elif sma_regime_ratio >= 1.05 and return_sma > 0:
        btc_regime = "BTC Bull"

    elif sma_selected_ratio <= 0.95 and return_sma < 0:
        btc_regime = "BTC Bear"

    elif sma_selected_ratio < 1.0 and return_momentum > 0:
        btc_regime = "BTC Recovery"

    else:
        btc_regime = "BTC Neutral"

    # ------------------------------------------------------------
    # 2. MVRV valuation regime
    # ------------------------------------------------------------
    if mvrv < 1.0:
        valuation_regime = "Low MVRV"

    elif mvrv > 2.5:
        valuation_regime = "High MVRV"

    else:
        valuation_regime = "Normal MVRV"

    # ------------------------------------------------------------
    # 3. Market-cap / realized-cap growth regime
    # ------------------------------------------------------------
    if realized_growth > market_growth:
        cap_regime = "Realized Growth Leading"

    else:
        cap_regime = "Market Growth Leading"

    # ------------------------------------------------------------
    # Final combined regime
    # ------------------------------------------------------------
    return btc_regime + " | " + valuation_regime + " | " + cap_regime


btc_data["combined_regime"] = btc_data.apply(
    classify_btc_mvrv_market_cap_regime,
    axis=1,
)

display(
    btc_data[["date", "price_usd", "combined_regime"]].head()
)

,date,price_usd,combined_regime
0,2011-08-16,11.05,BTC Severe Drawdown | Normal MVRV | Realized G...
1,2011-08-17,10.88,BTC Severe Drawdown | Normal MVRV | Realized G...
2,2011-08-18,10.90,BTC Severe Drawdown | Normal MVRV | Realized G...
3,2011-08-19,11.40,BTC Severe Drawdown | Normal MVRV | Realized G...
4,2011-08-20,11.49,BTC Severe Drawdown | Normal MVRV | Realized G...


In [7]:
# ============================================================
# Cell 7: Train/test split
# ============================================================
# This cell splits data into train and test periods.
# It also trims each split into complete 365-day evaluation windows.

raw_train_df = btc_data[
    (btc_data["date"] >= pd.to_datetime(TRAIN_START)) &
    (btc_data["date"] <= pd.to_datetime(TRAIN_END))
].copy().reset_index(drop=True)

raw_test_df = btc_data[
    (btc_data["date"] >= pd.to_datetime(TEST_START)) &
    (btc_data["date"] <= pd.to_datetime(TEST_END))
].copy().reset_index(drop=True)

train_eval_df, n_train_windows, train_remainder = trim_full_windows(raw_train_df, WINDOW_SIZE)
test_eval_df, n_test_windows, test_remainder = trim_full_windows(raw_test_df, WINDOW_SIZE)

split_summary_df = pd.DataFrame([
    {
        "split": "train",
        "start_date": raw_train_df["date"].min(),
        "end_date": raw_train_df["date"].max(),
        "rows_total": len(raw_train_df),
        "rows_eval": len(train_eval_df),
        "windows": n_train_windows,
        "remainder_dropped": train_remainder,
        "budget_rule": "$1,000 per 365-day training window",
    },
    {
        "split": "test",
        "start_date": raw_test_df["date"].min(),
        "end_date": raw_test_df["date"].max(),
        "rows_total": len(raw_test_df),
        "rows_eval": len(test_eval_df),
        "windows": n_test_windows,
        "remainder_dropped": test_remainder,
        "budget_rule": "$1,000 per 365-day test window",
    },
])

display(split_summary_df)

,split,start_date,end_date,rows_total,rows_eval,windows,remainder_dropped,budget_rule
0,train,2018-01-01,2023-12-31,2191,2190,6,1,"$1,000 per 365-day training window"
1,test,2024-01-01,2025-12-31,731,730,2,1,"$1,000 per 365-day test window"


In [8]:
# ============================================================
# Cell 7A: XGBoost target and feature preparation
# ============================================================
# This cell creates:
# 1. A training dataframe that includes the target label.
# 2. A scoring dataframe that only requires feature columns.
#
# This avoids using future-return target availability when generating predictions.

xgb_feature_cols = [
    "sma_ratio_feature",
    "momentum_btc_price_return",
    "sma_period_btc_price_return",
    "drawdown_feature",
    "mvrv",
    "adjusted_sopr",
    "adjusted_sopr_7d_ema",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

# Continuous future-return target.
# This is used only to create the supervised learning label.
btc_data["future_return_30d"] = (
    btc_data["price_usd"].shift(-XGB_FORWARD_HORIZON) / btc_data["price_usd"] - 1.0
)

# Create a rolling 70th-percentile threshold for future returns.
# This defines what counts as a "strong" future 30-day return.
# The shift by XGB_FORWARD_HORIZON + 1 avoids look-ahead leakage by using
# only future-return outcomes that would already be known at the current date.

btc_data["future_return_rolling_q70"] = (
    btc_data["future_return_30d"]
    .rolling(XGB_TARGET_ROLLING_WINDOW, min_periods=120)
    .quantile(XGB_TARGET_QUANTILE)
    .shift(XGB_FORWARD_HORIZON + 1)
)

# Binary classification target.
# 1 = future 30-day return is stronger than the recent rolling 70th percentile.
# 0 = otherwise.
btc_data["xgb_good_buy_day"] = (
    btc_data["future_return_30d"] > btc_data["future_return_rolling_q70"]
).astype(int)

# ------------------------------------------------------------
# Training dataframe
# ------------------------------------------------------------
# This dataframe is used to train XGBoost.
# It must have both features and target labels.
xgb_model_df = (
    btc_data
    .dropna(
        subset=xgb_feature_cols
        + ["future_return_30d", "future_return_rolling_q70", "xgb_good_buy_day"]
    )
    .sort_values("date")
    .reset_index(drop=True)
)

xgb_train_df = xgb_model_df[
    (xgb_model_df["date"] >= pd.to_datetime(TRAIN_START)) &
    (xgb_model_df["date"] <= pd.to_datetime(TRAIN_END))
].copy().reset_index(drop=True)

xgb_test_df = xgb_model_df[
    (xgb_model_df["date"] >= pd.to_datetime(TEST_START)) &
    (xgb_model_df["date"] <= pd.to_datetime(TEST_END))
].copy().reset_index(drop=True)

# ------------------------------------------------------------
# Scoring dataframe
# ------------------------------------------------------------
# This dataframe is used only for prediction/scoring.
# It only requires feature columns, not future-return target columns.
# This makes the prediction step cleaner and closer to a live strategy.
xgb_score_input_df = (
    btc_data
    .dropna(subset=xgb_feature_cols)
    .sort_values("date")
    .reset_index(drop=True)
)

print("XGBoost model rows with labels:", len(xgb_model_df))
print("XGBoost scoring rows with features only:", len(xgb_score_input_df))
print("XGBoost train rows:", len(xgb_train_df))
print("XGBoost test rows:", len(xgb_test_df))
print("XGBoost target positive rate in train:", round(xgb_train_df["xgb_good_buy_day"].mean(), 4))
print("XGBoost target positive rate in test:", round(xgb_test_df["xgb_good_buy_day"].mean(), 4))
print("XGBoost features:", xgb_feature_cols)

display(xgb_feature_description_df)

XGBoost model rows with labels: 5144
XGBoost scoring rows with features only: 5324
XGBoost train rows: 2191
XGBoost test rows: 731
XGBoost target positive rate in train: 0.3181
XGBoost target positive rate in test: 0.2353
XGBoost features: ['sma_ratio_feature', 'momentum_btc_price_return', 'sma_period_btc_price_return', 'drawdown_feature', 'mvrv', 'adjusted_sopr', 'adjusted_sopr_7d_ema', 'realized_cap_growth_rate', 'market_cap_growth_rate']


,xgb_feature_name,source_column,source_config,lookback_days,meaning
0,sma_ratio_feature,price_90d_sma_ratio,SMA_LOOKBACK,90.0,Price divided by SMA. Below 1 means BTC is tra...
1,momentum_btc_price_return,btc_return_60d,MOMENTUM_LOOKBACK,60.0,BTC price return over the momentum lookback pe...
2,sma_period_btc_price_return,btc_return_90d,SMA_LOOKBACK,90.0,BTC price return over the same period used for...
3,drawdown_feature,drawdown_180d,DRAWDOWN_LOOKBACK,180.0,Current price drawdown from the rolling high o...
4,mvrv,mvrv,on-chain metric,NaN,Valuation metric comparing market value to rea...
5,adjusted_sopr,adjusted_sopr,on-chain metric,NaN,Spent output profit ratio. Values below 1 can ...
6,adjusted_sopr_7d_ema,adjusted_sopr_7d_ema,on-chain metric,7.0,Smoothed SOPR signal.
7,realized_cap_growth_rate,realized_cap_growth_rate,on-chain metric,NaN,Growth rate of realized capitalization.
8,market_cap_growth_rate,market_cap_growth_rate,market metric,NaN,Growth rate of market capitalization.


In [9]:
# ============================================================
# Cell 7B: Train XGBoost and generate predictions
# ============================================================
# This cell trains the XGBoost classifier on the training period only.
# It then generates predicted probabilities for both train and test dates.

X_train_xgb = xgb_train_df[xgb_feature_cols].copy()
y_train_xgb = xgb_train_df["xgb_good_buy_day"].astype(int).copy()

if len(X_train_xgb) == 0:
    raise ValueError("No XGBoost training rows available. Check target and feature preparation.")

# Use a chronological validation split for monitoring.
split_idx = int(len(X_train_xgb) * 0.80)

X_xgb_train_fit = X_train_xgb.iloc[:split_idx].copy()
y_xgb_train_fit = y_train_xgb.iloc[:split_idx].copy()

X_xgb_val = X_train_xgb.iloc[split_idx:].copy()
y_xgb_val = y_train_xgb.iloc[split_idx:].copy()

xgb_model = XGBClassifier(
    n_estimators=250,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.80,
    colsample_bytree=0.80,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(
    X_xgb_train_fit,
    y_xgb_train_fit,
    eval_set=[(X_xgb_val, y_xgb_val)],
    verbose=False,
)


# Generate probabilities for all rows that have feature columns.
# This does not require future-return labels.
xgb_scored_df = xgb_score_input_df[["date"]].copy()

xgb_scored_df["xgboost_probability"] = xgb_model.predict_proba(
    xgb_score_input_df[xgb_feature_cols]
)[:, 1]

display(xgb_scored_df.head())
display(xgb_scored_df.tail())


# Feature importance table for interpretation.
xgb_feature_importance_df = pd.DataFrame({
    "feature": xgb_feature_cols,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

display(xgb_feature_importance_df.round(6))

,date,xgboost_probability
0,2011-08-16,0.008955
1,2011-08-17,0.008955
2,2011-08-18,0.010375
3,2011-08-19,0.010053
4,2011-08-20,0.007813


,date,xgboost_probability
5319,2026-03-09,0.329065
5320,2026-03-10,0.334026
5321,2026-03-11,0.196680
5322,2026-03-12,0.196680
5323,2026-03-13,0.196680


,feature,importance
7,realized_cap_growth_rate,0.222307
2,sma_period_btc_price_return,0.127668
8,market_cap_growth_rate,0.126170
4,mvrv,0.124772
0,sma_ratio_feature,0.113320
1,momentum_btc_price_return,0.111544
3,drawdown_feature,0.103553
6,adjusted_sopr_7d_ema,0.043593
5,adjusted_sopr,0.027074
